In [0]:
%run ../config

In [0]:
# Create fact_orders gold table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_catalog}.fact_orders
USING DELTA
AS
SELECT 
  o.order_id,
  cu.customer_key,
  COALESCE(br.branch_key, 0) AS branch_key,
  o.order_date,
  o.total_basket
FROM {silver_catalog}.orders_clean o
LEFT JOIN {gold_catalog}.dim_customers cu
  ON o.customer_id = cu.customer_id
LEFT JOIN {gold_catalog}.dim_branches br
  ON o.branch_id = br.branch_id
""")

# optimize for query performance
spark.sql(f"""
OPTIMIZE {gold_catalog}.fact_orders
ZORDER BY (order_date)
""")